# Capstone Project - Initial Report and EDA

## Predicting the direction of the US Treasury 10-year yield

**Arun Pandey** | PGP in AI and Machine Learning | Assignment 20.1

---

### My research question

**Can the shape of the US Treasury yield curve, together with policy and market data, predict
whether the 10-year yield will be higher or lower one month from now - better than just
guessing that it keeps doing what it has been doing?**

### Why I picked this

The US Treasury yield curve is used to price almost everything else in finance - corporate
bonds, swaps, mortgages, and also Indian government bonds, which is the market I work in.
When yields go up, bond prices go down, and banks holding a lot of bonds lose money. This is
what happened to Silicon Valley Bank in March 2023.

So the question is a real one that people on trading desks care about.

### What I expect

I am not expecting great accuracy. The Treasury market is one of the most watched markets in
the world, so if there was an easy pattern someone would have traded it away already. I will
be happy with something a bit better than a coin flip, and I want to measure it honestly
rather than get a big number by making a mistake.

### Dataset

[US Treasury Yields](https://www.kaggle.com/datasets/guillemservera/us-treasury-yields-daily)
from Kaggle (by Guillem SD, licence CC BY-NC 4.0). It has daily yields for 11 different
maturities from 1 month to 30 years. I picked it because I need all the maturities on the
same days to study the shape of the curve - most bond datasets only give one maturity.

I also pull a few extra series from FRED (the St. Louis Fed database) for context.

### Steps in this notebook

1. Load the data
2. Clean it - missing values, duplicates
3. Look for outliers
4. Make new features
5. Explore the data with charts
6. Build a baseline model
7. Conclusions

## 1. Load the data

In [ ]:
# Libraries I need
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 30)

print("Libraries loaded")

In [ ]:
# Download the dataset from Kaggle.
# kagglehub can usually download public datasets without needing a login.
import kagglehub, glob, os

path = kagglehub.dataset_download("guillemservera/us-treasury-yields-daily")
csv_file = glob.glob(os.path.join(path, "*.csv"))[0]

raw = pd.read_csv(csv_file)
print("File:", os.path.basename(csv_file))
print("Shape:", raw.shape)
print("Columns:", raw.columns.tolist())
raw.head()

In [ ]:
# The column names might be written a few different ways depending on the file
# (like 'US10Y' or '10 Yr'), so I clean them up into a simple format like '10Y'.
def tidy_name(name):
    name = str(name).upper().replace(" ", "")
    name = name.replace("US", "").replace("DGS", "")
    name = name.replace("YEARS", "Y").replace("YEAR", "Y").replace("YR", "Y")
    name = name.replace("MONTHS", "M").replace("MONTH", "M").replace("MO", "M")
    return name

# Find the date column and use it as the index
date_col = [c for c in raw.columns if "DATE" in str(c).upper()][0]
df = raw.copy()
df[date_col] = pd.to_datetime(df[date_col])
df = df.set_index(date_col).sort_index()
df.index.name = "date"

# Rename the yield columns
df.columns = [tidy_name(c) for c in df.columns]

# Keep only the 11 maturities I want, in order from shortest to longest
tenors = ["1M", "3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
tenors = [t for t in tenors if t in df.columns]
df = df[tenors]

print("Maturities found:", tenors)
print("Date range:", df.index.min().date(), "to", df.index.max().date())
df.tail()

In [ ]:
# I also want some extra variables that might help explain yield moves.
# These come from FRED and download as CSV without needing an API key.
fred_series = {
    "fed_funds": "DFF",            # Fed policy rate
    "breakeven": "T10YIE",         # market expectation of inflation
    "vix": "VIXCLS",               # stock market volatility index
    "oil": "DCOILBRENTEU",         # Brent crude oil price
    "dollar": "DTWEXBGS",          # US dollar index
    "cpi": "CPIAUCSL",             # inflation (monthly)
}

extras = {}
for name, code_id in fred_series.items():
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + code_id
    series = pd.read_csv(url)
    series.columns = ["date", name]
    series["date"] = pd.to_datetime(series["date"])
    # FRED writes missing values as "." so I convert those to NaN
    series[name] = pd.to_numeric(series[name], errors="coerce")
    extras[name] = series.set_index("date")[name]
    print("Downloaded", code_id)

extras = pd.DataFrame(extras)
extras.tail(3)

## 2. Clean the data

Three things to check: missing values, duplicates, and whether the dates line up.

In [ ]:
# How many missing values are there in each column?
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "first_date": [df[c].first_valid_index() for c in df.columns],
})
missing

In [ ]:
# Plot the missing values so I can see the pattern
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(df.columns, df.isna().mean() * 100, color="steelblue")
axes[0].set_title("Percentage of missing values by maturity")
axes[0].set_xlabel("Maturity")
axes[0].set_ylabel("Missing (%)")

# Resample to monthly so the picture is readable over 60 years
sns.heatmap(df.isna().resample("MS").mean().T, cmap="Reds", ax=axes[1],
            cbar_kws={"label": "Fraction missing"})
axes[1].set_title("When the missing values happen")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Maturity")
axes[1].set_xticks(axes[1].get_xticks()[::60])

plt.tight_layout()
plt.show()

The missing values are not random. Some maturities were simply not published in the early
years - the 1-month yield only starts in 2001 - and the 20-year bond was stopped for a few
years in the late 1980s.

So I will handle it in two ways:

- **Big structural gaps:** just start my analysis in August 2001, when all 11 maturities exist.
- **Small one-day gaps:** these are market holidays, so I fill them with the previous day's
  value. That makes sense because if a bond did not trade, yesterday's price is the best guess.

In [ ]:
rows_before = len(df)

# Step 1: only use dates from Aug 2001 onwards (all 11 maturities available)
df = df.loc["2001-08-01":]
rows_after_cut = len(df)

# Step 2: fill small gaps with the previous value.
# limit=3 stops it from filling a long gap by accident.
missing_before = df.isna().sum().sum()
df = df.ffill(limit=3)
missing_after = df.isna().sum().sum()

# Step 3: drop any date that is still incomplete (PCA later needs full rows)
df = df.dropna()

print("Rows at the start:      ", rows_before)
print("After cutting to 2001+: ", rows_after_cut)
print("Values filled in:       ", missing_before - missing_after)
print("Rows dropped:           ", rows_after_cut - len(df))
print("Rows I am left with:    ", len(df))

In [ ]:
# Check for duplicates
dup_dates = df.index.duplicated().sum()
dup_rows = df.duplicated().sum()

print("Duplicate dates:", dup_dates)
print("Rows where every value is identical to another row:", dup_rows)

if dup_dates > 0:
    df = df[~df.index.duplicated(keep="last")]
    print("Removed the duplicate dates. Rows now:", len(df))

# I am keeping the identical rows. If the whole curve prints the same on two days
# that is just a very quiet market, not a data error.

In [ ]:
# Put the extra FRED variables onto the same dates as the yield data
extras = extras.reindex(df.index.union(extras.index)).ffill().reindex(df.index)

# CPI is monthly and gets published about 2 weeks after the month it describes.
# If I used it on the month it refers to, my model would be seeing data that had not
# actually been published yet. That is called look-ahead bias, so I shift it forward.
extras["cpi_yoy"] = extras["cpi"].pct_change(252) * 100
extras["cpi_yoy"] = extras["cpi_yoy"].shift(15, freq="D").reindex(df.index).ffill()
extras = extras.drop(columns=["cpi"])

print("Extra variables:", extras.columns.tolist())
print("\nMissing values in extras (%):")
print((extras.isna().mean() * 100).round(1))

### Checking the downloads actually worked

Web downloads sometimes come back truncated without raising any error. If one of these series
only has a few hundred recent rows instead of its full history, then when I drop incomplete
rows later that one column decides how much data I keep - and I could lose most of it without
noticing, because nothing crashes.

So I check how much of my date range each series actually covers, and stop the notebook if one
looks wrong. Better to fail here than to fit a model on a tiny sample and believe the result.

In [ ]:
# What fraction of my dates does each series cover? The two that start late
# (breakeven in 2003, dollar in 2006) still cover most of the range. Anything
# under half has not downloaded properly.
coverage = extras.notna().mean()

for name in extras.columns:
    values = extras[name].dropna()
    flag = "" if coverage[name] >= 0.50 else "   <-- TOO SHORT"
    if len(values) == 0:
        print(f"{name:<12}{0:>7} rows{0:>8.1%}  EMPTY{flag}")
        continue
    print(f"{name:<12}{len(values):>7} rows{coverage[name]:>8.1%}  "
          f"{values.index.min().date()} to {values.index.max().date()}{flag}")

print()
if (coverage < 0.50).any():
    raise ValueError(
        "These series cover less than half my dates: "
        + str(coverage[coverage < 0.50].index.tolist())
        + ". The download was truncated - re-run the download cell above.")
print("All downloads look complete.")

**A variable I had to drop.** My first version also downloaded the high yield credit spread
(`BAMLH0A0HYM2`) as a second fear gauge. It kept downloading truncated - a few hundred recent
rows instead of the full history - and because that column was then missing for every year
before 2023, dropping incomplete rows threw away 97% of my data. I ended up with 40 rows in my
test set and a model that scored a perfect ROC-AUC of 1.00, which is what a broken result looks
like. I took the variable out and added the check above so it cannot happen quietly again.

Two of the extra variables start later than my yield data (the breakeven inflation rate starts
in 2003 and the dollar index in 2006). I will leave them in and drop the incomplete rows later,
after I have built all the features.

## 3. Outliers

I am looking at **daily changes** rather than the yield level. A 5% yield was normal in 2007
and very high in 2021, so the levels are not comparable across the years, but the daily
changes are.

In [ ]:
# Daily changes in basis points (1 basis point = 0.01%)
changes = df.diff().dropna() * 100

# Method 1: z-score. Anything more than 3 standard deviations from the mean.
z = (changes - changes.mean()) / changes.std()
z_outliers = (z.abs() > 3).sum()

# Method 2: IQR rule. Outside 1.5 x the interquartile range.
q1 = changes.quantile(0.25)
q3 = changes.quantile(0.75)
iqr = q3 - q1
iqr_outliers = ((changes < q1 - 1.5 * iqr) | (changes > q3 + 1.5 * iqr)).sum()

outlier_table = pd.DataFrame({
    "z_score_outliers": z_outliers,
    "iqr_outliers": iqr_outliers,
    "biggest_fall_bp": changes.min().round(1),
    "biggest_rise_bp": changes.max().round(1),
})
outlier_table

In [ ]:
# Boxplot to see the outliers
plt.figure(figsize=(11, 5))
sns.boxplot(data=changes, color="lightsteelblue")
plt.title("Daily yield changes by maturity (dots are outliers)")
plt.xlabel("Maturity")
plt.ylabel("Daily change (basis points)")
plt.axhline(0, color="black", linestyle="--", linewidth=0.8)
plt.show()

In [ ]:
# Which days were the biggest moves in the 10-year yield?
biggest = changes["10Y"].abs().nlargest(10).index
biggest_table = pd.DataFrame({
    "change_bp": changes.loc[biggest, "10Y"].round(1),
    "yield_after": df.loc[biggest, "10Y"].round(2),
}).sort_index()
biggest_table

In [ ]:
# Are the big moves spread out, or bunched into particular years?
outliers_per_year = (z["10Y"].abs() > 3).groupby(z.index.year).sum()
outliers_per_year = outliers_per_year[outliers_per_year > 0]

plt.figure(figsize=(11, 4))
plt.bar(outliers_per_year.index.astype(str), outliers_per_year.values, color="indianred")
plt.title("Number of extreme days in the 10-year yield, by year")
plt.xlabel("Year")
plt.ylabel("Number of days")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**My decision: keep the outliers.**

They are clustered in years I recognise - 2008 (financial crisis), 2020 (Covid) and 2022-23
(when the Fed raised rates very fast). That tells me these are real market events, not typing
errors in the data.

If I deleted them I would be removing exactly the interesting part of the data, and my model
would think big moves never happen. That is the mistake that made 2022 so painful for banks
holding bonds.

## 4. Explore the data

In [ ]:
# How have the yields moved over time?
plt.figure(figsize=(12, 5))
for t in ["3M", "2Y", "10Y", "30Y"]:
    plt.plot(df.index, df[t], label=t, linewidth=1)
plt.title("US Treasury yields over time")
plt.xlabel("Date")
plt.ylabel("Yield (%)")
plt.legend(title="Maturity")
plt.show()

In [ ]:
# The 10Y minus 2Y spread is the "slope" of the curve.
# When it goes below zero the curve is inverted, which people watch closely.
spread = df["10Y"] - df["2Y"]

plt.figure(figsize=(12, 4))
plt.plot(spread.index, spread, color="darkred", linewidth=1)
plt.fill_between(spread.index, spread, 0, where=(spread < 0), color="red", alpha=0.3,
                 label="Inverted")
plt.axhline(0, color="black", linestyle="--", linewidth=0.8)
plt.title("Slope of the yield curve (10-year minus 2-year)")
plt.xlabel("Date")
plt.ylabel("Spread (percentage points)")
plt.legend()
plt.show()

print("Curve was inverted on", (spread < 0).sum(), "days out of", len(spread),
      "(" + str(round((spread < 0).mean() * 100, 1)) + "%)")

In [ ]:
# Are the daily changes normally distributed? I will compare against a normal curve.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, t in zip(axes, ["3M", "5Y", "10Y"]):
    data = changes[t]
    ax.hist(data, bins=100, density=True, color="steelblue", alpha=0.7)

    # Draw a normal distribution with the same mean and standard deviation
    x = np.linspace(data.quantile(0.001), data.quantile(0.999), 200)
    normal = (1 / (data.std() * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - data.mean()) / data.std()) ** 2)
    ax.plot(x, normal, color="red", linewidth=2, label="Normal curve")

    ax.set_xlim(data.quantile(0.002), data.quantile(0.998))
    ax.set_title("Daily change: " + t)
    ax.set_xlabel("Change (basis points)")
    ax.set_ylabel("Density")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics of the daily changes
stats = pd.DataFrame({
    "mean": changes.mean(),
    "std_dev": changes.std(),
    "skew": changes.skew(),
    "kurtosis": changes.kurtosis(),
    "min": changes.min(),
    "max": changes.max(),
}).round(2)
stats

The kurtosis is much higher than 0 for every maturity. Kurtosis measures how fat the tails
are, and 0 would mean a normal distribution. So the real data has far more extreme days than
a bell curve would predict.

This matters because a lot of statistical methods assume normal errors. If I use those, I will
underestimate how often a big loss can happen.

In [ ]:
# How correlated are the different maturities with each other?
plt.figure(figsize=(9, 7))
sns.heatmap(changes.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=0, vmax=1,
            square=True, cbar_kws={"label": "Correlation"})
plt.title("Correlation of daily yield changes between maturities")
plt.show()

# Average correlation, ignoring the diagonal of 1s
corr_values = changes.corr().values
avg_corr = corr_values[np.triu_indices_from(corr_values, k=1)].mean()
print("Average correlation between maturities:", round(avg_corr, 3))

The correlations are very high - the maturities mostly move together. This is why PCA is going
to be useful. Instead of 11 columns that all say roughly the same thing, I can compress them
into a few factors that capture the real movement.

In [ ]:
# An interactive chart, so I can hover over it and compare the curve shape on different dates.
# I picked one date from each of a few different periods.
dates_to_show = []
for year in [2007, 2012, 2019, 2021, 2023]:
    year_data = df.loc[str(year)]
    if len(year_data) > 0:
        dates_to_show.append(year_data.index[len(year_data) // 2])
dates_to_show.append(df.index[-1])

snapshot = df.loc[dates_to_show].T
snapshot.columns = [d.strftime("%b %Y") for d in dates_to_show]
snapshot = snapshot.reset_index().melt(id_vars="index", var_name="Date", value_name="Yield")
snapshot = snapshot.rename(columns={"index": "Maturity"})

fig = px.line(snapshot, x="Maturity", y="Yield", color="Date", markers=True,
              title="Shape of the yield curve on different dates")
fig.update_layout(xaxis_title="Maturity", yaxis_title="Yield (%)", height=450)
fig.show()

## 5. Feature engineering

Now I build the columns I will actually give to the model. Raw yield levels are not great
inputs because they drift over time, so I mostly use differences and spreads.

### 5.1 PCA - compressing the curve into 3 factors

In [ ]:
# I run PCA on the daily CHANGES, not the levels.
# If I used levels, the first component would just track the long slow fall in interest
# rates since 2001, which is true but not useful for predicting anything.
scaled_changes = StandardScaler().fit_transform(changes)

pca = PCA(n_components=3, random_state=42)
pc_values = pca.fit_transform(scaled_changes)
pc = pd.DataFrame(pc_values, index=changes.index, columns=["PC1", "PC2", "PC3"])

explained = pca.explained_variance_ratio_ * 100
print("PC1 explains", round(explained[0], 1), "% of the variation")
print("PC2 explains", round(explained[1], 1), "%")
print("PC3 explains", round(explained[2], 1), "%")
print("Together:", round(explained.sum(), 1), "%")

In [ ]:
# The loadings tell me what each component actually represents
loadings = pd.DataFrame(pca.components_.T, index=changes.columns,
                        columns=["PC1", "PC2", "PC3"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].bar(["PC1", "PC2", "PC3"], explained, color="steelblue")
for i, v in enumerate(explained):
    axes[0].text(i, v + 1, str(round(v, 1)) + "%", ha="center")
axes[0].set_title("How much each component explains")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Variance explained (%)")
axes[0].set_ylim(0, 105)

for col, style in zip(loadings.columns, ["o-", "s-", "^-"]):
    axes[1].plot(loadings.index, loadings[col], style, label=col, linewidth=2)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("What each component means")
axes[1].set_xlabel("Maturity")
axes[1].set_ylabel("Loading")
axes[1].legend()

plt.tight_layout()
plt.show()
loadings.round(3)

This matches what the textbooks say about yield curves:

- **PC1** is roughly flat across all maturities, so it means the whole curve moving up or
  down together. This is called the **level**.
- **PC2** goes steadily from negative at the short end to positive at the long end, so it
  is the curve getting steeper or flatter. This is the **slope**.
- **PC3** is negative in the middle and positive at both ends - a hump shape. This is the
  **curvature**.

Rather than just saying the chart looks right, I checked it with numbers below.

In [ ]:
# Checking the shapes with numbers instead of just eyeballing the chart
maturity_years = [1/12, 0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]

# PC1: if it is flat, the standard deviation should be small compared to the average
pc1_flatness = loadings["PC1"].std() / abs(loadings["PC1"].mean())

# PC2: if it increases with maturity, rank correlation should be close to +1 or -1
pc2_trend = pd.Series(loadings["PC2"].values).corr(pd.Series(maturity_years), method="spearman")

# PC3: a hump changes sign twice (positive, negative, positive)
pc3_sign_changes = (np.diff(np.sign(loadings["PC3"].values)) != 0).sum()

print("PC1 flatness (closer to 0 = flatter):     ", round(pc1_flatness, 3))
print("PC2 trend with maturity (want near +/-1): ", round(pc2_trend, 3))
print("PC3 number of sign changes (want 2):      ", pc3_sign_changes)

### 5.2 Finding market regimes with k-means

In [ ]:
# I use k-means on the PCA scores to group days into "regimes".
# First I smooth the scores, because day-to-day noise is too jumpy for this.
pc_smooth = pc.rolling(21).mean().dropna()

kmeans = KMeans(n_clusters=4, n_init=20, random_state=42)
labels = kmeans.fit_predict(StandardScaler().fit_transform(pc_smooth))
regime = pd.Series(labels, index=pc_smooth.index, name="regime")

# The cluster numbers come out in a random order, so I renumber them
# from lowest average yield to highest. That makes them easier to talk about.
avg_yield = df["10Y"].reindex(regime.index).groupby(regime).mean()
new_numbers = {old: new for new, old in enumerate(avg_yield.sort_values().index)}
regime = regime.map(new_numbers)

regime_summary = pd.DataFrame({
    "days": regime.value_counts().sort_index(),
    "avg_10y_yield": df["10Y"].reindex(regime.index).groupby(regime).mean().round(2),
    "avg_slope": spread.reindex(regime.index).groupby(regime).mean().round(2),
})
regime_summary

In [ ]:
# Plot the regimes against the 10-year yield to see if they make sense
yield_10y = df["10Y"].reindex(regime.index)

plt.figure(figsize=(13, 4.5))
plt.plot(yield_10y.index, yield_10y, color="black", linewidth=1, zorder=3, label="10Y yield")
colors = ["#a6cee3", "#b2df8a", "#fdbf6f", "#fb9a99"]
for r in sorted(regime.unique()):
    plt.fill_between(regime.index, yield_10y.min() - 0.3, yield_10y.max() + 0.3,
                     where=(regime == r), color=colors[r], alpha=0.5, step="mid",
                     label="Regime " + str(r))
plt.ylim(yield_10y.min() - 0.2, yield_10y.max() + 0.2)
plt.title("Market regimes found by k-means, shown against the 10-year yield")
plt.xlabel("Date")
plt.ylabel("Yield (%)")
plt.legend(ncol=5, fontsize=9)
plt.show()

The regimes line up with periods I recognise - the low-rate years after 2008, and the sharp
rise in 2022-23. The model was never told about the Fed or any of those events, it found the
groups just from the shape of the curve, so that is a good sign the factors mean something
real.

### 5.3 Building the feature table

In [ ]:
features = pd.DataFrame(index=df.index)

# Shape of the curve
features["level_10y"] = df["10Y"]
features["slope_10y_2y"] = df["10Y"] - df["2Y"]
features["slope_10y_3m"] = df["10Y"] - df["3M"]
features["slope_30y_10y"] = df["30Y"] - df["10Y"]
features["butterfly"] = 2 * df["5Y"] - df["2Y"] - df["10Y"]

# Carry = how much extra you earn holding a 10-year bond instead of a 3-month bill
features["carry"] = df["10Y"] - df["3M"]
features["rolldown"] = df["10Y"] - df["7Y"]

# Momentum - how much the yield has moved recently
for days in [1, 5, 21]:
    features["change_" + str(days) + "d"] = df["10Y"].diff(days)
features["gap_from_average"] = df["10Y"] - df["10Y"].rolling(63).mean()
features["volatility"] = df["10Y"].diff().rolling(21).std() * np.sqrt(252)

# Policy and inflation
features["fed_funds"] = extras["fed_funds"]
features["fed_funds_change"] = extras["fed_funds"].diff(63)
features["yield_minus_policy"] = df["10Y"] - extras["fed_funds"]
features["breakeven"] = extras["breakeven"]
features["real_yield"] = df["10Y"] - extras["breakeven"]
features["cpi_yoy"] = extras["cpi_yoy"]

# Risk and markets
features["vix"] = extras["vix"]
features["dollar"] = extras["dollar"]
features["oil_return"] = extras["oil"].pct_change(21)

# Categorical features
features["regime"] = regime.reindex(features.index).ffill()

# Is the curve inverted, flat or normal? This is how traders describe it.
features["curve_shape"] = pd.cut(features["slope_10y_2y"], bins=[-np.inf, 0, 0.5, np.inf],
                                 labels=["Inverted", "Flat", "Normal"])

# Split volatility into three equal-sized groups
features["vol_level"] = pd.qcut(features["volatility"], q=3,
                                labels=["Low", "Medium", "High"])

print("Number of features:", features.shape[1])
features.tail(3).T.head(12)

In [ ]:
# The target: will the 10-year yield be higher in 21 trading days (about 1 month)?
# shift(-21) looks FORWARD in time, which is what I want to predict.
future_change = df["10Y"].shift(-21) - df["10Y"]

data = features.copy()
data["change_bp"] = future_change * 100
data["target"] = (future_change > 0).astype(int)   # 1 = yield goes up

# Before dropping, see what each column costs me in rows
rows_before = len(data)
cost = data.isna().sum()
cost = cost[cost > 0].sort_values(ascending=False)
if len(cost) > 0:
    print("Rows each column would cost me:")
    print(cost.to_string())
    print()

data = data.dropna()
data["regime"] = data["regime"].astype(int)

# Losing the rolling-window warm-up is normal. Losing more than half is not.
if len(data) < 0.5 * rows_before:
    raise ValueError(
        "Dropping missing values removed more than half the rows (" + str(rows_before)
        + " -> " + str(len(data)) + "). See the table above for which column caused it.")

print("Final dataset:", data.shape[0], "rows and", features.shape[1], "features")
print("Date range:", data.index.min().date(), "to", data.index.max().date())
print("Target balance:", round(data["target"].mean() * 100, 1), "% of the time yields went up")

### Checking I have not cheated

This is the part I was most careful about. It is very easy to accidentally give the model
information it would not have had at the time, and then the results look great but are
meaningless.

- The target uses `shift(-21)`, so it is definitely looking forward.
- All rolling averages look backwards only - I never used `center=True`.
- CPI is shifted by 15 days so the model does not see it before it was published.
- All the columns come from the same US market on the same day, so there is no time zone
  problem.

One thing I should be honest about: the regime labels came from k-means fitted on the whole
dataset, so they have seen a bit of the future. I test the model with and without that column
in the next notebook to check it is not doing the work.

## 6. Exploring the categorical features

I made three categorical columns - the regime, the curve shape, and the volatility level.
Here I check whether they actually tell me anything about the target.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

sns.countplot(data=data, x="regime", ax=axes[0], color="steelblue")
axes[0].set_title("Days in each regime")
axes[0].set_xlabel("Regime")
axes[0].set_ylabel("Number of days")

sns.countplot(data=data, x="curve_shape", ax=axes[1], color="mediumseagreen",
              order=["Inverted", "Flat", "Normal"])
axes[1].set_title("Days by curve shape")
axes[1].set_xlabel("Curve shape")
axes[1].set_ylabel("Number of days")

sns.countplot(data=data, x="vol_level", ax=axes[2], color="orange",
              order=["Low", "Medium", "High"])
axes[2].set_title("Days by volatility level")
axes[2].set_xlabel("Volatility")
axes[2].set_ylabel("Number of days")

target_labels = data["target"].map({0: "Down", 1: "Up"})
sns.countplot(x=target_labels, ax=axes[3], color="indianred", order=["Down", "Up"])
axes[3].set_title("Target: did the yield go up?")
axes[3].set_xlabel("Direction after 21 days")
axes[3].set_ylabel("Number of days")

plt.tight_layout()
plt.show()

In [ ]:
# Does the future move look different depending on the category?
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.boxplot(data=data, x="regime", y="change_bp", ax=axes[0], color="steelblue")
axes[0].set_title("Future yield change by regime")
axes[0].set_xlabel("Regime")
axes[0].set_ylabel("Change over next 21 days (bp)")

sns.boxplot(data=data, x="curve_shape", y="change_bp", ax=axes[1],
            order=["Inverted", "Flat", "Normal"], color="mediumseagreen")
axes[1].set_title("Future yield change by curve shape")
axes[1].set_xlabel("Curve shape")
axes[1].set_ylabel("Change over next 21 days (bp)")

sns.boxplot(data=data, x="vol_level", y="change_bp", ax=axes[2],
            order=["Low", "Medium", "High"], color="orange")
axes[2].set_title("Future yield change by volatility")
axes[2].set_xlabel("Volatility level")
axes[2].set_ylabel("Change over next 21 days (bp)")

for ax in axes:
    ax.axhline(0, color="black", linestyle="--", linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# What percentage of the time did yields go up, within each category?
# If a category is different from the overall average, it might be useful.
overall = data["target"].mean() * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
columns = ["regime", "curve_shape", "vol_level"]
orders = [None, ["Inverted", "Flat", "Normal"], ["Low", "Medium", "High"]]

for ax, col, order in zip(axes, columns, orders):
    pct_up = data.groupby(col, observed=True)["target"].mean() * 100
    if order:
        pct_up = pct_up.reindex(order)
    ax.bar(pct_up.index.astype(str), pct_up.values, color="steelblue")
    ax.axhline(overall, color="red", linestyle="--", linewidth=2,
               label="Overall average (" + str(round(overall, 1)) + "%)")
    for i, v in enumerate(pct_up.values):
        ax.text(i, v + 1, str(round(v, 1)) + "%", ha="center", fontsize=9)
    ax.set_title("Yields went up: by " + col)
    ax.set_xlabel(col)
    ax.set_ylabel("Percentage of days (%)")
    ax.set_ylim(0, 100)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Do any single features have a strong relationship with the future change?
check_features = ["slope_10y_2y", "carry", "change_21d", "volatility"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.flat, check_features):
    ax.scatter(data[col], data["change_bp"], s=4, alpha=0.25, color="steelblue")

    # Draw a straight line of best fit
    slope, intercept = np.polyfit(data[col], data["change_bp"], 1)
    x_line = np.linspace(data[col].min(), data[col].max(), 100)
    ax.plot(x_line, slope * x_line + intercept, color="red", linewidth=2)

    r = data[col].corr(data["change_bp"])
    ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
    ax.set_title(col + "  (correlation = " + str(round(r, 3)) + ")")
    ax.set_xlabel(col)
    ax.set_ylabel("Future change (bp)")

plt.tight_layout()
plt.show()

All the correlations are small. I was expecting this - if one single number could predict
where bond yields were going, people would have found it and traded on it already.

It does tell me something useful though: no single feature is going to work on its own, so if
there is anything here it has to come from combining several weak signals.

## 7. Baseline model

I am using **logistic regression** because:

- The target is yes/no (does the yield go up?), so this is a classification problem.
- The coefficients are easy to read, so I can see which features matter.
- It is a simple model, which is what a baseline should be. If I try something fancier later,
  I need to show it actually beats this.

In [ ]:
# Turn the text categories into numbers that sklearn can use
model_data = pd.get_dummies(data, columns=["curve_shape", "vol_level"],
                            drop_first=True, dtype=float)

X = model_data.drop(columns=["target", "change_bp"])
y = model_data["target"]

# Split by DATE, not randomly. Randomly splitting time series data lets the model
# train on the future and test on the past, which makes the results look far better
# than they really are.
split_point = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_point], X.iloc[split_point:]
y_train, y_test = y.iloc[:split_point], y.iloc[split_point:]

print("Training:", X_train.index.min().date(), "to", X_train.index.max().date(),
      "(", len(X_train), "rows )")
print("Testing: ", X_test.index.min().date(), "to", X_test.index.max().date(),
      "(", len(X_test), "rows )")
print("Features:", X.shape[1])

### Which measure am I using to judge the model?

**My main measure is ROC-AUC.** Reasons:

1. What I really want from the model is a *confidence level* - how sure is it that yields go
   up - because a trader would use that to decide how big a position to take. ROC-AUC measures
   how well the model ranks the days, which is exactly that. Accuracy throws the ranking away
   by forcing everything into yes/no at a cut-off of 0.5.
2. It does not depend on where I set that cut-off, and 0.5 is just a default, not a decision.
3. It does not get flattered by having more of one class than the other.

**I also report accuracy against two benchmarks**, because my research question is literally
about beating a random walk:

- **Majority class**: always predict whichever answer was more common. If I cannot beat this,
  I have learned nothing at all.
- **Persistence**: predict that the yield keeps moving the way it moved last month. This is the
  "random walk" from my research question.

How to read the ROC-AUC number: 0.5 is a coin flip. For predicting bond yields a month ahead,
0.55 to 0.65 would be a small but real edge. If I got something above 0.70 I would assume I
had made a mistake somewhere and go looking for it.

In [ ]:
# The two benchmarks I have to beat
majority = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
majority_pred = majority.predict(X_test)

# Persistence: assume the next month's move has the same sign as the last month's
persistence_pred = (X_test["change_21d"] > 0).astype(int)

print("Majority class  - accuracy:", round(accuracy_score(y_test, majority_pred), 4))
print("Persistence     - accuracy:", round(accuracy_score(y_test, persistence_pred), 4),
      " ROC-AUC:", round(roc_auc_score(y_test, persistence_pred), 4))

In [ ]:
# Build the model.
# I put the scaler inside a Pipeline so that it only learns from the training data
# in each fold. If I scaled everything up front, information from the test period
# would leak into training.
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=3000, random_state=42))
])

# TimeSeriesSplit instead of normal cross validation - each fold trains on the past
# and tests on the period straight after it.
cv = TimeSeriesSplit(n_splits=5)

grid = GridSearchCV(model, {"logreg__C": [0.01, 0.05, 0.1, 0.5, 1, 5]},
                    cv=cv, scoring="roc_auc")
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("Best C value:", grid.best_params_["logreg__C"])
print("Cross-validation ROC-AUC:", round(grid.best_score_, 4))

In [ ]:
# Now test it on the held-out period. I only do this once.
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

results = pd.DataFrame([
    {"model": "Logistic regression",
     "accuracy": accuracy_score(y_test, y_pred),
     "precision": precision_score(y_test, y_pred, zero_division=0),
     "recall": recall_score(y_test, y_pred, zero_division=0),
     "f1": f1_score(y_test, y_pred, zero_division=0),
     "roc_auc": roc_auc_score(y_test, y_prob)},
    {"model": "Persistence (random walk)",
     "accuracy": accuracy_score(y_test, persistence_pred),
     "precision": precision_score(y_test, persistence_pred, zero_division=0),
     "recall": recall_score(y_test, persistence_pred, zero_division=0),
     "f1": f1_score(y_test, persistence_pred, zero_division=0),
     "roc_auc": roc_auc_score(y_test, persistence_pred)},
    {"model": "Majority class",
     "accuracy": accuracy_score(y_test, majority_pred),
     "precision": 0, "recall": 0, "f1": 0, "roc_auc": 0.5},
]).set_index("model").round(4)
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0].plot(fpr, tpr, linewidth=2, color="steelblue",
             label="My model (AUC = " + str(round(roc_auc_score(y_test, y_prob), 3)) + ")")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="Random guessing (AUC = 0.5)")
axes[0].set_title("ROC curve on the test set")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].legend()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1],
            xticklabels=["Predicted down", "Predicted up"],
            yticklabels=["Actually down", "Actually up"])
axes[1].set_title("Confusion matrix")

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=["Down", "Up"], digits=3))

In [ ]:
# Which features did the model rely on?
# The coefficients are comparable because everything was scaled first.
coefs = pd.Series(best_model.named_steps["logreg"].coef_[0], index=X_train.columns)
top_15 = coefs.sort_values(key=abs, ascending=False).head(15).sort_values()

plt.figure(figsize=(9, 6))
plt.barh(top_15.index, top_15.values,
         color=["indianred" if v > 0 else "steelblue" for v in top_15.values])
plt.axvline(0, color="black", linewidth=1)
plt.title("Top 15 features in the logistic regression")
plt.xlabel("Coefficient (positive = pushes towards yields going up)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 8. Conclusions

In [ ]:
# Print a summary I can copy into my README
auc = roc_auc_score(y_test, y_prob)
acc = accuracy_score(y_test, y_pred)
persistence_acc = accuracy_score(y_test, persistence_pred)

print("SUMMARY OF RESULTS")
print("=" * 55)
print("Rows of clean data:      ", len(df))
print("Date range:              ", df.index.min().date(), "to", df.index.max().date())
print("Rows used for modelling: ", len(data))
print("Number of features:      ", X.shape[1])
print()
print("PCA: first 3 components explain", round(explained.sum(), 1), "% of daily variation")
print("     (level =", round(explained[0], 1), "%, slope =", round(explained[1], 1),
      "%, curvature =", round(explained[2], 1), "%)")
print()
print("Baseline model ROC-AUC:  ", round(auc, 4))
print("Baseline model accuracy: ", round(acc, 4))
print("Persistence accuracy:    ", round(persistence_acc, 4))
print("Improvement:             ", round((acc - persistence_acc) * 100, 2), "percentage points")
print()

if auc > 0.70:
    print("VERDICT: That is suspiciously high for this problem. I should check for a mistake")
    print("         before believing it.")
elif auc >= 0.55:
    print("VERDICT: A small but possibly real edge, which is what I expected.")
elif auc > 0.52:
    print("VERDICT: Barely better than guessing. Any edge here is very small.")
else:
    print("VERDICT: No real predictive power. On this evidence the market is efficient")
    print("         at a one month horizon, which is a valid finding.")

### What I found

1. **The yield curve is basically 3-dimensional.** Even though I have 11 maturities, three
   PCA components explain nearly all the daily movement, and they match the level, slope and
   curvature idea from the literature. I checked this with numbers, not just by looking at
   the chart.

2. **Yield changes are not normally distributed.** The kurtosis is high for every maturity,
   meaning extreme days happen much more often than a bell curve says. Any risk model built
   on a normal assumption would underestimate how bad a bad day can get.

3. **The big moves are real events, not errors.** They cluster in 2008, 2020 and 2022-23,
   which is why I kept them rather than removing them.

4. **No single feature predicts the target.** All the correlations with the future change are
   small, which is what I would expect in a market this heavily traded.

5. **The regimes found by k-means make sense.** They line up with periods I recognise, even
   though the algorithm was never told about the Fed or any events.

### What I would do differently / next

- Try other models (KNN, decision tree, SVM) and see whether any of them beat this baseline.
  That is the next notebook.
- Show properly how much a normal random train/test split would have inflated my results,
  because I think it is the most important thing I learned here.
- Test whether the model behaves differently before and after 2008, since interest rates were
  stuck near zero for years after the financial crisis and that period may just be a different
  world.
- Check whether the regime column is doing real work, by running the model without it.